> **SwiftMSeg: Lightweight Multi-Scale Local–Global Context Modeling with Transformer for Medical Image Segmentation**

In [ ]:
import os
import random
import numpy as np
import cv2
from glob import glob
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import SegformerForSemanticSegmentation

import torchvision
import torchvision.transforms as T

# import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, jaccard_score


torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
! pip install thop

In [ ]:
# dataset loading + pre-processing

images_dir = ""
masks_dir  = ""

class SegmentationDataset(Dataset):

    def __init__(self, images_dir, masks_dir, classes_to_keep, file_list, transform=None):
        self.transform = transform
        self.classes_to_keep = classes_to_keep
    
        self.mapping = {v: i for i, v in enumerate(classes_to_keep)}
        
    
        self.images_paths = [
            os.path.join(images_dir, f"{fname}.jpg") for fname in file_list
        ]
        self.masks_paths = [
            os.path.join(masks_dir, f"{fname}_mask.png") for fname in file_list
        ]
        assert len(self.images_paths) == len(self.masks_paths)
        
    def __len__(self):
        return len(self.images_paths)
    
    def __getitem__(self, idx):
        image = cv2.imread(self.images_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(self.masks_paths[idx], cv2.IMREAD_GRAYSCALE)
        

        mask_remap = np.zeros_like(mask)
        for orig_val, new_idx in self.mapping.items():
            mask_remap[mask == orig_val] = new_idx
        mask = mask_remap
        
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        
        return image, mask.long()
def get_transforms():
    train_transform = A.Compose([
        A.Resize(256, 256),
        # A.HorizontalFlip(p=0.5),
        # A.VerticalFlip(p=0.5),
        # A.RandomRotate90(p=0.5),
        # A.RandomBrightnessContrast(p=0.2),
        A.Normalize((0.485,0.456,0.406), (0.229,0.224,0.225)),
        ToTensorV2()
    ])
    val_transform = A.Compose([
        A.Resize(256, 256),
        A.Normalize((0.485,0.456,0.406), (0.229,0.224,0.225)),
        ToTensorV2()
    ])
    return train_transform, val_transform

train_transform, val_transform = get_transforms()

classes_to_keep = [0, 1]  # ******** loading appropriate semantic class labels from the dataset

# get all basenames
all_files = [
    os.path.basename(p).replace('.jpg','')
    for p in sorted(glob(os.path.join(images_dir, '*.jpg')))
]


random.seed(42)
random.shuffle(all_files)
n = len(all_files)
print(n)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)
train_files = all_files[:n_train]
val_files   = all_files[n_train:n_train+n_val]
test_files  = all_files[n_train+n_val:]
print(f" train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

# create datasets
train_ds = SegmentationDataset(images_dir, masks_dir, classes_to_keep, train_files, transform=train_transform)
val_ds   = SegmentationDataset(images_dir, masks_dir, classes_to_keep, val_files,   transform=val_transform)
test_ds  = SegmentationDataset(images_dir, masks_dir, classes_to_keep, test_files,  transform=val_transform)

# dataloaders
batch_size = 1
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}, Test samples: {len(test_ds)}")
num_classes = len(classes_to_keep)
print ("Num of Class: ", num_classes)

######################### Main **Architecture**  #####################################

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# LGL Block: Local–Global–Local Feature Aggregation
# -----------------------------
class LGLBlock(nn.Module):
    """Local-Global-Local block: depthwise conv -> multi-head self-attention -> depthwise conv."""
    def __init__(self, dim, num_heads=4):
        super().__init__()
        # 1. Local projection (depthwise convolution) to capture local context
        self.local1 = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
        self.norm1 = nn.LayerNorm(dim)  # LayerNorm for transformer (applied to channel dim)

        # 2. Global self-attention (MultiheadAttention) to capture long-range context
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)

        # 3. Local projection (another depthwise conv) for further local refinement
        self.local2 = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)

    def forward(self, x):
        B, C, H, W = x.shape

        # Local conv (depthwise) on feature map
        x_local = self.local1(x)  # shape: [B, C, H, W]

        # Flatten spatial dimensions for attention: [B, C, H, W] -> [B, H*W, C]
        x_flat = x_local.permute(0, 2, 3, 1).reshape(B, H * W, C)
        # Normalize before attention
        x_norm = self.norm1(x_flat)

        # Global self-attention (batch_first=True so B is first dim)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)  # self-attention on flattened tokens
        # Add skip-connection (residual) from input to attention output
        x_attn = x_norm + attn_out
        x_attn = self.norm2(x_attn)  # normalize after attention

        # Reshape back to [B, C, H, W]
        x_attn = x_attn.reshape(B, H, W, C).permute(0, 3, 1, 2)

        # Second local conv for refinement
        x_out = self.local2(x_attn)
        return x_out


# -----------------------------
class ProposedModel(nn.Module):
    def __init__(self, num_classes=6, in_channels=3):
        super().__init__()
        # Encoder: Hierarchical downsampling with overlapping convs + LGL blocks at each stage

        # Stage 1: Patch embedding (overlapping conv) to 1/4 resolution
        self.patch_embed = nn.Conv2d(in_channels, 32, kernel_size=7, stride=4, padding=3)
        self.lgl1 = LGLBlock(dim=32, num_heads=4)   # Local-Global block for 1/4 scale features

        # Stage 2: Downsample to 1/8 resolution
        self.down2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.lgl2 = LGLBlock(dim=64, num_heads=4)   # LGL block for 1/8 scale features

        # Stage 3: Downsample to 1/16 resolution
        self.down3 = nn.Conv2d(64, 160, kernel_size=3, stride=2, padding=1)
        self.lgl3 = LGLBlock(dim=160, num_heads=4)  # LGL block for 1/16 scale features

        # Stage 4: Downsample to 1/32 resolution
        self.down4 = nn.Conv2d(160, 256, kernel_size=3, stride=2, padding=1)
        self.lgl4 = LGLBlock(dim=256, num_heads=4)  # LGL block for 1/32 scale features

        # Decoder: Up-sampling path (UNet-style)
        # Upsample deeper features and concatenate with correspondingly scaled encoder feature
        self.up3 = nn.ConvTranspose2d(256, 160, kernel_size=2, stride=2)        # upsample from 1/32 -> 1/16
        self.dec3 = nn.Conv2d(160 + 160, 160, kernel_size=3, padding=1)         # fuse with stage3 (1/16 features)

        self.up2 = nn.ConvTranspose2d(160, 64, kernel_size=2, stride=2)         # 1/16 -> 1/8
        self.dec2 = nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1)            # fuse with stage2 (1/8 features)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)          # 1/8 -> 1/4
        self.dec1 = nn.Conv2d(32 + 32, 32, kernel_size=3, padding=1)            # fuse with stage1 (1/4 features)

        self.final = nn.Conv2d(32, num_classes, kernel_size=1)  # output segmentation logits

    def forward(self, x):
        # Encoder forward pass
        # Stage 1
        x1 = self.patch_embed(x)       # [B, 32, H/4, W/4]
        e1 = self.lgl1(x1)             # apply local-global feature refinement

        # Stage 2
        x2 = self.down2(e1)           # [B, 64, H/8, W/8]
        e2 = self.lgl2(x2)

        # Stage 3
        x3 = self.down3(e2)           # [B, 160, H/16, W/16]
        e3 = self.lgl3(x3)

        # Stage 4
        x4 = self.down4(e3)           # [B, 256, H/32, W/32]
        e4 = self.lgl4(x4)

        # Decoder forward pass (UNet-style upsampling and concatenation)
        d3 = self.up3(e4)                             # Upsample stage4 output to 1/16
        d3 = torch.cat([d3, e3], dim=1)               # Concatenate with stage3 features
        d3 = F.relu(self.dec3(d3))                    # Decode convolution at 1/16 scale

        d2 = self.up2(d3)                             # Upsample to 1/8
        d2 = torch.cat([d2, e2], dim=1)               # Concatenate with stage2 features
        d2 = F.relu(self.dec2(d2))                    # Decode convolution at 1/8 scale

        d1 = self.up1(d2)                             # Upsample to 1/4
        d1 = torch.cat([d1, e1], dim=1)               # Concatenate with stage1 features
        d1 = F.relu(self.dec1(d1))                    # Decode convolution at 1/4 scale

        out = self.final(d1)                          # [B, num_classes, H/4, W/4] raw logits
        # Upsample final logits back to original image size
        out = F.interpolate(out, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)
        return out  # output logits (apply softmax or argmax outside as needed)

# -----------------------------
# Metrics (multiclass segmentation)
# -----------------------------
def pixel_accuracy(preds, labels):
    """Overall pixel-wise accuracy."""
    preds = preds.argmax(dim=1)
    return (preds == labels).float().mean().item()

def dice_score(preds, labels, num_classes):
    """Mean Dice coefficient (multiclass)"""
    preds = preds.argmax(dim=1)
    smooth = 1e-6
    dice = 0.0
    for cls in range(num_classes):
        pred_cls = (preds == cls).float()
        label_cls = (labels == cls).float()
        inter = (pred_cls * label_cls).sum()
        union = pred_cls.sum() + label_cls.sum()
        dice += (2 * inter + smooth) / (union + smooth)
    return dice / num_classes

def iou_score(preds, labels, num_classes):
    """Mean Intersection-over-Union (IoU)"""
    preds = preds.argmax(dim=1)
    smooth = 1e-6
    iou = 0.0
    for cls in range(num_classes):
        pred_cls = (preds == cls).float()
        label_cls = (labels == cls).float()
        inter = (pred_cls * label_cls).sum()
        union = pred_cls.sum() + label_cls.sum() - inter
        iou += (inter + smooth) / (union + smooth)
    return iou / num_classes

# -----------------------------
# Example: Model instantiation (for 6 classes as in thesis)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ProposedModel(num_classes=6).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


In [ ]:
# # -----------------------------
# # Model: SegFormer encoder + UNet-style decoder
# # -----------------------------
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import SegformerModel

# # -----------------------------
# # LGL Block
# # -----------------------------
# class LGLBlock(nn.Module):
#     """Local-Global-Local block: local conv -> shallow transformer -> local conv"""
#     def __init__(self, dim, num_heads=4):
#         super().__init__()
#         self.local1 = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
#         self.norm1 = nn.LayerNorm(dim)

#         self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
#         self.norm2 = nn.LayerNorm(dim)

#         self.local2 = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)

#     def forward(self, x):
#         B, C, H, W = x.shape

#         # Local conv
#         x = self.local1(x)

#         # Flatten for attention
#         x_flat = x.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         x_norm = self.norm1(x_flat)

#         # Global self-attention
#         attn_out, _ = self.attn(x_norm, x_norm, x_norm)
#         x_flat = x_norm + attn_out
#         x_flat = self.norm2(x_flat)

#         # Reshape back
#         x_out = x_flat.reshape(B, H, W, C).permute(0, 3, 1, 2)

#         # Local conv
#         x_out = self.local2(x_out)
#         return x_out


# # -----------------------------
# # SegFormer Encoder + LGL + U-Net Decoder
# # -----------------------------
# from transformers import SegformerModel

# class SegFormerMobileUVit(nn.Module):
#     def __init__(self, num_classes=6, backbone="nvidia/mit-b0"):  #Use Pretrained Weight
#         super().__init__()
#         # Segformer backbone (encoder only)
#         self.backbone = SegformerModel.from_pretrained(backbone, output_hidden_states=True)

#         # Add LGL refinement after backbone stages
#         self.lgl_blocks = nn.ModuleList([
#             LGLBlock(dim) for dim in [32, 64, 160, 256]  # mit-b0 channel dims
#         ])

#         # U-Net style decoder
#         self.up3 = nn.ConvTranspose2d(256, 160, 2, stride=2)
#         self.dec3 = nn.Conv2d(160 + 160, 160, 3, padding=1)

#         self.up2 = nn.ConvTranspose2d(160, 64, 2, stride=2)
#         self.dec2 = nn.Conv2d(64 + 64, 64, 3, padding=1)

#         self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
#         self.dec1 = nn.Conv2d(32 + 32, 32, 3, padding=1)

#         self.final = nn.Conv2d(32, num_classes, 1)

#     def forward(self, x):
#         outputs = self.backbone(x, output_hidden_states=True)
#         feats = outputs.hidden_states  # 4 feature maps

#         # Apply LGL blocks
#         feats = [lgl(f) for lgl, f in zip(self.lgl_blocks, feats)]
#         e1, e2, e3, e4 = feats

#         # Decoder
#         d3 = self.up3(e4)
#         d3 = torch.cat([d3, e3], dim=1)
#         d3 = F.relu(self.dec3(d3))

#         d2 = self.up2(d3)
#         d2 = torch.cat([d2, e2], dim=1)
#         d2 = F.relu(self.dec2(d2))

#         d1 = self.up1(d2)
#         d1 = torch.cat([d1, e1], dim=1)
#         d1 = F.relu(self.dec1(d1))

#         out = self.final(d1)
#         out = F.interpolate(out, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)
#         return out  # logits (no softmax)


# # -----------------------------
# # Metrics (multiclass)
# # -----------------------------
# def pixel_accuracy(preds, labels):
#     preds = preds.argmax(dim=1)
#     return (preds == labels).float().mean().item()

# def dice_score(preds, labels, num_classes):
#     preds = preds.argmax(dim=1)
#     dice = 0
#     smooth = 1e-6
#     for cls in range(num_classes):
#         pred_cls = (preds == cls).float()
#         label_cls = (labels == cls).float()
#         inter = (pred_cls * label_cls).sum()
#         union = pred_cls.sum() + label_cls.sum()
#         dice += (2 * inter + smooth) / (union + smooth)
#     return dice / num_classes

# def iou_score(preds, labels, num_classes):
#     preds = preds.argmax(dim=1)
#     iou = 0
#     smooth = 1e-6
#     for cls in range(num_classes):
#         pred_cls = (preds == cls).float()
#         label_cls = (labels == cls).float()
#         inter = (pred_cls * label_cls).sum()
#         union = pred_cls.sum() + label_cls.sum() - inter
#         iou += (inter + smooth) / (union + smooth)
#     return iou / num_classes

# # -----------------------------
# # Training
# # -----------------------------
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = SegFormerMobileUVit(num_classes=num_classes).to(device)

# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
num_epochs = 50
history = {"loss": [], "val_loss": [], "val_dice": [], "val_iou": [], "val_acc": []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # validation
    model.eval()
    val_loss, val_dice, val_iou, val_acc = 0, 0, 0, 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = criterion(logits, masks)
            val_loss += loss.item()
            val_dice += dice_score(logits, masks, num_classes).item()
            val_iou += iou_score(logits, masks, num_classes).item()
            val_acc += pixel_accuracy(logits, masks)

    avg_train_loss = running_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_val_dice = val_dice / len(val_loader)
    avg_val_iou = val_iou / len(val_loader)
    avg_val_acc  = val_acc / len(val_loader)

    history["loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_dice"].append(avg_val_dice)
    history["val_iou"].append(avg_val_iou)
    history["val_acc"].append(avg_val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Dice: {avg_val_dice:.4f} | "
          f"Val IoU: {avg_val_iou:.4f} | "
          f"Val Acc: {avg_val_acc:.4f}")

# -----------------------------
# Final Test Evaluation
# -----------------------------
model.eval()
test_dice, test_iou, test_acc = 0, 0, 0
with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        test_dice += dice_score(logits, masks, num_classes).item()
        test_iou  += iou_score(logits, masks, num_classes).item()
        test_acc  += pixel_accuracy(logits, masks)

test_dice /= len(test_loader)
test_iou  /= len(test_loader)
test_acc  /= len(test_loader)

print(f"Test Dice: {test_dice:.4f}, Test IoU: {test_iou:.4f}, Test Acc: {test_acc:.4f}")

**Result Comparison: ** 
* default MiT-B0: *0.7270 0.6589 0.9112*
* Proposed Model: *0.7868 0.7488 0.9052*  (pre-trained)
* 
                  * *0.7308, Test IoU: 0.6949, Test Acc: 0.8625**

In [ ]:
from thop import profile

import time

# =========================
# Instantiate your model
# =========================
# model = ProposedModel(num_classes=6).cuda()
# model.eval()

# =========================
# 1. Params and FLOPs
# =========================
input_tensor = torch.randn(1, 3, 256, 256).cuda()
macs, params = profile(model, inputs=(input_tensor,), verbose=False)

gflops = (macs * 2) / 1e9  # MACs × 2 = FLOPs → GFLOPs
params_m = params / 1e6    # Params → Millions

# =========================
# 2. Latency and Throughput
# =========================
def benchmark(model, input_tensor, warmup=10, runs=50):
    timings = []
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_tensor)

        torch.cuda.synchronize()
        for _ in range(runs):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            _ = model(input_tensor)
            end.record()

            torch.cuda.synchronize()
            elapsed = start.elapsed_time(end)  # milliseconds
            timings.append(elapsed)

    avg_latency = sum(timings) / len(timings)
    throughput = 1000.0 / avg_latency  # images/sec
    return avg_latency, throughput

latency_ms, throughput = benchmark(model, input_tensor)

# =========================
# 3. Print Results
# =========================
print(f"\n{'Model':30} {'Params(M)':>10} {'GFLOPs':>10} {'Latency(ms)':>15} {'Throughput(img/s)':>20}")
print("-" * 95)
print(f"{'ProposedModel':30} {params_m:10.3f} {gflops:10.3f} {latency_ms:15.2f} {throughput:20.2f}")


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total trainable parameters: {count_parameters(model)/ 1e6 :,}")
